# 02 — Feature Engineering
IEEE-CIS Fraud Detection

In [ ]:
import pandas as pd
import numpy as np

# Load data
trans = pd.read_csv('../data/raw/train_transaction.csv', low_memory=False)
identity = pd.read_csv('../data/raw/train_identity.csv', low_memory=False)
df = trans.merge(identity, on='TransactionID', how='left')

In [ ]:
# 1. Log transform transaction amount
df['amt_log'] = np.log1p(df['TransactionAmt'])

In [ ]:
# 2. Time features from TransactionDT
df['hour'] = (df['TransactionDT'] / 3600).astype(int) % 24
df['day']  = (df['TransactionDT'] / 86400).astype(int) % 7

In [ ]:
# 3. Card velocity features
df['card_txn_count'] = df.groupby('card1')['TransactionID'].transform('count')
df['card_avg_amt']   = df.groupby('card1')['TransactionAmt'].transform('mean')
df['card_std_amt']   = df.groupby('card1')['TransactionAmt'].transform('std')
df['amt_vs_card_avg'] = df['TransactionAmt'] / (df['card_avg_amt'] + 1)

In [ ]:
# 4. Client UID trick (D1 - TransactionDT = near constant per client)
df['client_uid'] = df['TransactionDT'] - (df['D1'] * 86400)

In [ ]:
# 5. Encode M columns (match flags) as 0/1/-1
m_cols = [c for c in df.columns if c.startswith('M')]
for col in m_cols:
    df[col] = df[col].map({'T': 1, 'F': 0}).fillna(-1)

In [ ]:
# 6. Drop V columns with more than 90% nulls
v_cols = [c for c in df.columns if c.startswith('V')]
v_null = df[v_cols].isnull().mean()
v_keep = v_null[v_null < 0.9].index.tolist()
print(f'Keeping {len(v_keep)} of {len(v_cols)} V columns')

In [ ]:
# 7. Save processed data
cols_to_keep = ['isFraud', 'amt_log', 'hour', 'day',
                'card_txn_count', 'card_avg_amt', 'card_std_amt',
                'amt_vs_card_avg', 'client_uid'] + m_cols + v_keep

df[cols_to_keep].to_parquet('../data/processed/features.parquet', index=False)
print('Saved to data/processed/features.parquet')